# DATA PROFILING


In [3]:
# ALL THE LIBRARIES AND PACKAGES INSTALLED
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [4]:
pd.set_option("display.max_columns",None) 
pd.set_option("display.max_rows",100)
pd.set_option("display.float_format" ,"{:,.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


In [5]:
# PROJECT PATHS.
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"
REPORTS_PATH = PROJECT_ROOT / "reports"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)
print("Processed data path:", PROCESSED_DATA_PATH)
print("Reports path:", REPORTS_PATH)

Project root: e:\Learning\Projects\Olist-Ecom
Raw data path: e:\Learning\Projects\Olist-Ecom\data\raw
Processed data path: e:\Learning\Projects\Olist-Ecom\data\processed
Reports path: e:\Learning\Projects\Olist-Ecom\reports


In [6]:
# CONFIRMING THE DATA FILES ARE FOUND
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError("Raw data folder not found: {RAW_DATA_PATH}")
else:
    print("Raw data folder found.")

Raw data folder found.


In [7]:
csvFiles = sorted(RAW_DATA_PATH.glob("*.csv"))
print(f"Number of CSV files found: {len(csvFiles)}")

for file in csvFiles:
    print(file.name)

Number of CSV files found: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [8]:
# DATASET CONFIGURATION CREATION
DATA_FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

print("Dataset configuration created.")

Dataset configuration created.


In [9]:
dataset = {}
for name, fileName in DATA_FILES.items():
    filePath = RAW_DATA_PATH / fileName
    if not filePath.exists():
        raise FileNotFoundError("Missing File {filePath}")
    dataset[name] = pd.read_csv(filePath)
    
print("All datasets loaded Successfully!!!")


All datasets loaded Successfully!!!


In [10]:
# x----- SHAPE OF THE DATASETS-------x
datasetShape = []
for name, df in dataset.items():
    datasetShape.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]    
        }
    )
# datasetShape
shapeDF = pd.DataFrame(datasetShape)
shapeDF

,dataset,rows,columns
0,customers,99441,5
1,geolocation,1000163,5
2,order_items,112650,7
3,order_payments,103886,5
4,order_reviews,99224,7
5,orders,99441,8
6,products,32951,9
7,sellers,3095,4
8,category_translation,71,2


In [11]:
# x------------- DATA TYPES OF THE DATASET ---------------x

for name, df in dataset.items():
    print("-"*30)
    print(name.upper())
    print(df.dtypes)


------------------------------
CUSTOMERS
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object
------------------------------
GEOLOCATION
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object
------------------------------
ORDER_ITEMS
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object
------------------------------
ORDER_PAYMENTS
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object
-----------------------------

In [12]:
# x----------- DATA DICTIONARY ---------------X

dataDictionary = []
for name, df in dataset.items():
    for col in df:
        dataDictionary.append({
                'dataset': name,
                'columns': col,
                'dataType': str(df[col].dtypes),
                'count': len(df[col]),
                'missingValues': df[col].isna().sum(),
                'missingPercentage': (df[col].isna().mean() * 100).round(2),
                'uniqueValues': (df[col].nunique(dropna = True))
            })
# dataDictionary
dataDictionaryDF = pd.DataFrame(dataDictionary)
dataDictionaryDF

,dataset,columns,dataType,count,missingValues,missingPercentage,uniqueValues
0,customers,customer_id,str,99441,0,0.00,99441
1,customers,customer_unique_id,str,99441,0,0.00,96096
2,customers,customer_zip_code_prefix,int64,99441,0,0.00,14994
3,customers,customer_city,str,99441,0,0.00,4119
4,customers,customer_state,str,99441,0,0.00,27
5,geolocation,geolocation_zip_code_prefix,int64,1000163,0,0.00,19015
6,geolocation,geolocation_lat,float64,1000163,0,0.00,717360
7,geolocation,geolocation_lng,float64,1000163,0,0.00,717613
8,geolocation,geolocation_city,str,1000163,0,0.00,8011
9,geolocation,geolocation_state,str,1000163,0,0.00,27


In [13]:
# ---------- CHECKING MISSINGS ----------
missingSummary = []

for name, df in dataset.items():
    for column in df.columns:
        missingSummary.append({
            "dataset": name,
            "column": column,
            "missingCount": df[column].isna().sum(),
            "missingPercentage": (df[column].isna().mean() * 100).round(2)
        })

missingDF = pd.DataFrame(missingSummary)

missingDF = missingDF[
    missingDF["missingCount"] > 0
].sort_values(
    "missingPercentage",
    ascending=False
)

missingDF


,dataset,column,missingCount,missingPercentage
25,order_reviews,review_comment_title,87656,88.34
26,order_reviews,review_comment_message,58247,58.70
35,orders,order_delivered_customer_date,2965,2.98
39,products,product_name_lenght,610,1.85
38,products,product_category_name,610,1.85
40,products,product_description_lenght,610,1.85
41,products,product_photos_qty,610,1.85
34,orders,order_delivered_carrier_date,1783,1.79
33,orders,order_approved_at,160,0.16
42,products,product_weight_g,2,0.01


In [14]:
# missingSummaryTest = [{'dataset': 'customers',
#   'column': 'customer_id',
#   'missingCount': np.int64(0),
#   'missingPercentage': np.float64(0.0)}]
missingColumnsList = []
count = 0
for data in missingSummary:
    for name, value in data.items():
        # print()
        if name == 'missingCount':
            if data[name] > 0:
                missingColumnsList.append(data['column'])
                print(f"Column {data['column']} added successfully!!")
                count = count + 1
print(f"{count} Missing Columns are added successfully!")

                


Column review_comment_title added successfully!!
Column review_comment_message added successfully!!
Column order_approved_at added successfully!!
Column order_delivered_carrier_date added successfully!!
Column order_delivered_customer_date added successfully!!
Column product_category_name added successfully!!
Column product_name_lenght added successfully!!
Column product_description_lenght added successfully!!
Column product_photos_qty added successfully!!
Column product_weight_g added successfully!!
Column product_length_cm added successfully!!
Column product_height_cm added successfully!!
Column product_width_cm added successfully!!
13 Missing Columns are added successfully!


In [15]:
missingColumnsList

['review_comment_title',
 'review_comment_message',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

In [16]:
#------ DUPLICATE IN KEYS CHECKING -----------X
PRIMARY_KEYS = {
    "orders": ["order_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    'test': None,
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["order_id","review_id"],
    "category_translation": ["product_category_name"],
    "geolocation": None
}

for name, keyCol in PRIMARY_KEYS.items():
    if keyCol is None:
        continue
    df = dataset[name]
    duplicateKey = df.duplicated(
        subset = keyCol
    ).sum()
    print(
        f"{name}: "
        f"{duplicateKey} duplicate primary-key combinations"
    )

orders: 0 duplicate primary-key combinations
customers: 0 duplicate primary-key combinations
products: 0 duplicate primary-key combinations
sellers: 0 duplicate primary-key combinations
order_items: 0 duplicate primary-key combinations
order_payments: 0 duplicate primary-key combinations
order_reviews: 0 duplicate primary-key combinations
category_translation: 0 duplicate primary-key combinations


In [17]:
order = dataset['orders']
order.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [18]:
order.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [19]:
order[(['order_status','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date',
        'order_estimated_delivery_date'
        ])]

,order_status,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,delivered,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,delivered,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,delivered,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,delivered,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,delivered,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...
99436,delivered,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,delivered,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,delivered,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,delivered,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [20]:
order.groupby('order_status')[[
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]].apply(lambda x: x.isna().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,
approved,0,2,2,0
canceled,141,550,619,0
created,5,5,5,0
delivered,14,2,8,0
invoiced,0,314,314,0
processing,0,301,301,0
shipped,0,0,1107,0
unavailable,0,609,609,0


In [21]:
missingDeliveryDate = (order.loc[(order['order_status']=='delivered') & 
                        order['order_delivered_customer_date'].isna()])

print(f"Anomolies: {len(missingDeliveryDate)} Missing Delivery Date in Delivered Orders")

Anomolies: 8 Missing Delivery Date in Delivered Orders


In [22]:
deliveredMask = (
    (order['order_status']=='delivered') &
    (order['order_delivered_customer_date'].notna())
)

print(f"Delivered Orders: {deliveredMask.sum()}")


Delivered Orders: 96470


In [23]:
orderEDA = order.copy()
dateColumns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in dateColumns:
    orderEDA[col] = pd.to_datetime(orderEDA[col], errors='coerce')

orderEDA[dateColumns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [24]:
def dateMinMax(df ,columns):
    for col in columns:
        print(f"{col} Minimum {df[col].min()}")
        print(f"{col} Maximum {df[col].max()}")

dateMinMax(orderEDA,dateColumns)

order_purchase_timestamp Minimum 2016-09-04 21:15:19
order_purchase_timestamp Maximum 2018-10-17 17:30:18
order_approved_at Minimum 2016-09-15 12:16:38
order_approved_at Maximum 2018-09-03 17:40:06
order_delivered_carrier_date Minimum 2016-10-08 10:34:01
order_delivered_carrier_date Maximum 2018-09-11 19:48:28
order_delivered_customer_date Minimum 2016-10-11 13:46:32
order_delivered_customer_date Maximum 2018-10-17 13:22:46
order_estimated_delivery_date Minimum 2016-09-30 00:00:00
order_estimated_delivery_date Maximum 2018-11-12 00:00:00


In [25]:
orderEDA['delivery_days'] = (
    orderEDA['order_delivered_customer_date']
    - orderEDA['order_purchase_timestamp']
).dt.total_seconds() / (24*60*60)

In [26]:
orderEDA['delivery_delay_days'] = (
    orderEDA['order_delivered_customer_date']
    - orderEDA['order_estimated_delivery_date']
).dt.total_seconds() / (24*60*60)

dateMinMax(orderEDA,['delivery_delay_days'])

delivery_delay_days Minimum -146.0161226851852
delivery_delay_days Maximum 188.97508101851852


In [27]:
lateDeliveredOrder = orderEDA.loc[
    deliveredMask & (orderEDA["delivery_delay_days"] > 0)
]
print(f"Shape Of LateDeliveryOrder {lateDeliveredOrder.shape}")

lateDeliveredRate = (
            (len(lateDeliveredOrder)) / (deliveredMask.sum())
)*100

print(lateDeliveredRate)

Shape Of LateDeliveryOrder (7826, 10)
8.112366538820359


Customer Inspection

In [28]:
customer = dataset['customers']
customer.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [29]:
print(f"Customer Shape {customer.shape}")
print(f"Customer Columns {customer.columns}")
print("Unique customer IDs:", customer["customer_unique_id"].nunique())
print("Customer IDs:", customer["customer_id"].nunique())

customerOrderCounts = (
    customer["customer_unique_id"]
    .value_counts()
)

Customer Shape (99441, 5)
Customer Columns Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='str')
Unique customer IDs: 96096
Customer IDs: 99441


In [30]:
customerOrderCounts = (
    customer["customer_unique_id"]
    .value_counts()
)

customerOrderCounts

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
                                    ..
1a29b476fee25c95fbafc67c5ac95cf8     1
d52a67c98be1cf6a5c84435bd38d095d     1
e9f50caf99f032f0bf3c55141f019d99     1
73c2643a0a458b49f58cea58833b192e     1
84732c5050c01db9b23e19ba39899398     1
Name: count, Length: 96096, dtype: int64

In [31]:
(customerOrderCounts > 1).sum()

np.int64(2997)

In [32]:
# Does every `customer_id` in Order exist in Customers
orderEDA['customer_id'].isin(customer['customer_id']).value_counts()

customer_id
True    99441
Name: count, dtype: int64

In [33]:
customerOrderEDA = orderEDA.merge(
                    customer[[
                        "customer_id",
                        "customer_state",
                        "customer_city"
                    ]],
                    on = "customer_id",
                    how = "left"
                )

customerOrderEDA.head(5)
# orderEDA.columns

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,customer_state,customer_city
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.44,-7.11,SP,sao paulo
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.78,-5.36,BA,barreiras
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.39,-17.25,GO,vianopolis
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.21,-12.98,RN,sao goncalo do amarante
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.87,-9.24,SP,santo andre


In [108]:
orderEDA.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
delivery_days                    2965
delivery_delay_days              2965
dtype: int64

In [34]:
(customerOrderEDA.loc[
    customerOrderEDA['delivery_days'].isna(), ['order_status','customer_state', 'delivery_days']
])

,order_status,customer_state,delivery_days
6,invoiced,RS,NaN
44,shipped,SP,NaN
103,invoiced,SC,NaN
128,processing,SP,NaN
154,shipped,MG,NaN
...,...,...,...
99283,canceled,SP,NaN
99313,processing,SP,NaN
99347,canceled,SP,NaN
99348,unavailable,RJ,NaN


In [35]:
overview =customerOrderEDA.groupby(['customer_state','order_status']).agg(
    totalOrder = ('order_status','count'),
    deliveryDaysMissing = ('delivery_days', lambda x: x.isna().sum())
)
print(overview)


                             totalOrder  deliveryDaysMissing
customer_state order_status                                 
AC             delivered             80                    0
               shipped                1                    1
AL             canceled               1                    1
               delivered            397                    0
               invoiced               2                    2
...                                 ...                  ...
TO             canceled               1                    1
               delivered            274                    0
               processing             1                    1
               shipped                3                    3
               unavailable            1                    1

[149 rows x 2 columns]


OrderItem Profiling

In [36]:
orderItems = dataset['order_items']
orderItems.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [37]:
print('OrderItem - Order Id: Duplicated:', orderItems['order_id'].duplicated().sum())
print('OrderItem - OrderItemId: Duplicated:', orderItems['order_item_id'].duplicated().sum())
print('Total Data: ', len(orderItems))

OrderItem - Order Id: Duplicated: 13984
OrderItem - OrderItemId: Duplicated: 112629
Total Data:  112650


In [38]:
orderItems.duplicated(
    subset=['order_id', 'order_item_id']
).sum()

np.int64(0)

In [39]:
orderItems.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')

In [40]:
orderItems['shipping_limit_date'] = pd.to_datetime(orderItems['shipping_limit_date'],errors='coerce')

In [41]:
orderItems.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

In [42]:
print('Price == 0:',(orderItems["price"] == 0).sum())
print('freight value < 0:',(orderItems["freight_value"] < 0).sum())


Price == 0: 0
freight value < 0: 0


In [43]:
orderItemsEDA = orderItems.copy()
orderItemsEDA['item_total'] = (orderItemsEDA['price'] + orderItemsEDA['freight_value'])
orderItemsEDA[[
    "order_id",
    "price",
    "freight_value",
    "item_total"
]].head()

,order_id,price,freight_value,item_total
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04


In [44]:
orderItems["order_id"].isin(orderEDA["order_id"]).value_counts()

order_id
True    112650
Name: count, dtype: int64

In [45]:
orderWithoutItem = orderEDA[~orderEDA['order_id'].isin(orderItems['order_id'])]
len(orderWithoutItem)
# orderWithoutItem.columns
# orderWithoutItem['order_status'].value_counts()
# orderWithoutItem

775

In [46]:
itemsPerOrder = (
    orderItemsEDA.groupby("order_id")["order_item_id"]
    .count()
)

itemsPerOrder

order_id
00010242fe8c5a6d1ba2dd792cb16214    1
00018f77f2f0320c557190d7a144bdd3    1
000229ec398224ef6ca0657da4fc703e    1
00024acbcdf0a6daa1e931b038114c75    1
00042b26cf59d7ce69dfabb4e55b4fd9    1
                                   ..
fffc94f6ce00a00581880bf54a75a037    1
fffcd46ef2263f404302a634eb57f7eb    1
fffce4705a9662cd70adb13d4a31832d    1
fffe18544ffabc95dfada21779c9644f    1
fffe41c64501cc87c801fd61db3f6244    1
Name: order_item_id, Length: 98666, dtype: int64

In [47]:
orderTotals = (
    orderItemsEDA.groupby("order_id")
    .agg(
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        total_value=("item_total", "sum")
    )
)

In [48]:
orderTotals = orderTotals.join(
    itemsPerOrder.rename("item_count")
)

orderTotals.head()

,total_price,total_freight,total_value,item_count
order_id,,,,
00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19,1
00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83,1
000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87,1
00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78,1
00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04,1


In [49]:
len(orderTotals)

98666

In [50]:
orderTotals

,total_price,total_freight,total_value,item_count
order_id,,,,
00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19,1
00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83,1
000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87,1
00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78,1
00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04,1
...,...,...,...,...
fffc94f6ce00a00581880bf54a75a037,299.99,43.41,343.40,1
fffcd46ef2263f404302a634eb57f7eb,350.00,36.53,386.53,1
fffce4705a9662cd70adb13d4a31832d,99.90,16.95,116.85,1


In [51]:
orderItemsEDA.loc[(orderItemsEDA['order_id']=="3a213fcdfe7d98be74ea0dc05a8b31ae"),['price']]

,price
25564,108.00
25565,108.00
25566,108.00
25567,108.00
25568,108.00
25569,108.00
25570,108.00
25571,108.00
25572,108.00
25573,108.00


In [52]:
orderCustomerSaleEDA = (customerOrderEDA.merge(
    orderTotals,
    on= 'order_id',
    how = 'left'
))


In [53]:
orderCustomerSaleEDA

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,customer_state,customer_city,total_price,total_freight,total_value,item_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.44,-7.11,SP,sao paulo,29.99,8.72,38.71,1.00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.78,-5.36,BA,barreiras,118.70,22.76,141.46,1.00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.39,-17.25,GO,vianopolis,159.90,19.22,179.12,1.00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.21,-12.98,RN,sao goncalo do amarante,45.00,27.20,72.20,1.00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.87,-9.24,SP,santo andre,19.90,8.72,28.62,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,8.22,-10.37,SP,sao jose dos campos,72.00,13.08,85.08,1.00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,22.19,-1.27,SP,praia grande,174.90,20.10,195.00,1.00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,24.86,-5.52,BA,nova vicosa,205.99,65.02,271.01,1.00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,17.09,-20.02,RJ,japuiba,359.98,81.18,441.16,2.00


In [54]:
orderCustomerSaleEDA['purchaseMonth'] = orderCustomerSaleEDA['order_purchase_timestamp'].dt.to_period("M")

Products Profiling

In [55]:
products = dataset["products"]

products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,"1,000.00",30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.00,261.00,1.00,371.00,26.00,4.00,26.00
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.00,402.00,4.00,625.00,20.00,17.00,13.00


In [56]:
len(products)

32951

In [57]:
products.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

In [58]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [59]:
products['product_category_name'] = products['product_category_name'].fillna('unknown')

In [60]:
# (products[products['product_category_name'].isna()])
products['product_id'].isin(orderItems['product_id']).sum()

np.int64(32951)

In [61]:
orderProduct = orderItemsEDA.merge(
    products[[
        "product_id",
        "product_category_name"
    ]],
    on = "product_id",
    how = "left"
)


orderProduct.head(5)
# orderEDA.columns

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,item_total,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87,moveis_decoracao
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78,perfumaria
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04,ferramentas_jardim


In [62]:
orderProduct['product_category_name'].isna().sum()

np.int64(0)

In [63]:
len(orderCustomerSaleEDA)

99441

In [64]:
orderProduct

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,item_total,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87,moveis_decoracao
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78,perfumaria
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04,ferramentas_jardim
...,...,...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,343.40,utilidades_domesticas
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,386.53,informatica_acessorios
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,116.85,esporte_lazer
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,64.71,informatica_acessorios


In [65]:
itemsPerOrder

order_id
00010242fe8c5a6d1ba2dd792cb16214    1
00018f77f2f0320c557190d7a144bdd3    1
000229ec398224ef6ca0657da4fc703e    1
00024acbcdf0a6daa1e931b038114c75    1
00042b26cf59d7ce69dfabb4e55b4fd9    1
                                   ..
fffc94f6ce00a00581880bf54a75a037    1
fffcd46ef2263f404302a634eb57f7eb    1
fffce4705a9662cd70adb13d4a31832d    1
fffe18544ffabc95dfada21779c9644f    1
fffe41c64501cc87c801fd61db3f6244    1
Name: order_item_id, Length: 98666, dtype: int64

In [66]:
# # orderEDA[orderEDA['order_id'].isin(orderProduct['order_id'])]
productDetails = orderProduct.merge(
    orderCustomerSaleEDA[[
        "order_id",
        "order_status",
        "customer_state",
        "customer_city"
    ]],
    on = "order_id",
    how = "left"
)
productDetails

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,item_total,product_category_name,order_status,customer_state,customer_city
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19,cool_stuff,delivered,RJ,campos dos goytacazes
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83,pet_shop,delivered,SP,santa fe do sul
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87,moveis_decoracao,delivered,MG,para de minas
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78,perfumaria,delivered,SP,atibaia
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04,ferramentas_jardim,delivered,SP,varzea paulista
...,...,...,...,...,...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,343.40,utilidades_domesticas,delivered,MA,sao luis
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,386.53,informatica_acessorios,delivered,PR,curitiba
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,116.85,esporte_lazer,delivered,SP,sao paulo
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,64.71,informatica_acessorios,delivered,SP,vinhedo


Reviews Profiling

In [67]:
reviews = dataset["order_reviews"]
reviews
# dataset.keys()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13


In [110]:
(reviews.groupby("order_id")["review_score"]
        .mean()
        .reset_index()
)

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.00
1,00018f77f2f0320c557190d7a144bdd3,4.00
2,000229ec398224ef6ca0657da4fc703e,5.00
3,00024acbcdf0a6daa1e931b038114c75,4.00
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.00
...,...,...
98668,fffc94f6ce00a00581880bf54a75a037,5.00
98669,fffcd46ef2263f404302a634eb57f7eb,5.00
98670,fffce4705a9662cd70adb13d4a31832d,5.00
98671,fffe18544ffabc95dfada21779c9644f,5.00


In [68]:
reviews.columns

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='str')

In [69]:
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')

In [70]:
# reviews[reviews["review_score"]==0]
# reviews["review_score"].between(1, 5)
invalidReviews = reviews[
    ~reviews["review_score"].between(1, 5)
]

print("Invalid review scores:", len(invalidReviews))

Invalid review scores: 0


Payments Profiling

In [71]:
payments = dataset["order_payments"]

payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [72]:
len(payments)

103886

In [73]:
multiPayments = (
    payments.groupby('order_id')
    .filter(lambda x: len(x) > 1)
)

In [74]:
multiPayments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
25,5cfd514482e22bc992e7693f0e3e8df7,2,voucher,1,45.17
35,b2bb080b6bc860118a246fd9b6fad6da,1,credit_card,1,173.84
75,3689194c14ad4e2e7361ebd1df0e77b0,2,voucher,1,57.53
84,723e462ce1ee50e024887c0b403130f3,1,credit_card,1,13.80
102,21b8b46679ea6482cbf911d960490048,2,voucher,1,43.12
...,...,...,...,...,...
103778,fd86c80924b4be8fb7f58c4ecc680dae,1,credit_card,1,76.10
103817,6d4616de4341417e17978fe57aec1c46,1,credit_card,1,19.18
103860,31bc09fdbd701a7a4f9b55b5955b8687,6,voucher,1,77.99
103869,c9b01bef18eb84888f0fd071b8413b38,1,credit_card,6,238.16


In [75]:
payments.duplicated(
    subset = ['order_id','payment_sequential']
).sum()

np.int64(0)

In [76]:
payments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


In [77]:
maxSplits = payments.groupby('order_id')['payment_sequential'].max().max()
maxSplits

np.int64(29)

In [78]:
# payments.columns
payments.groupby('order_id').agg(
        sequentialCount = ('payment_sequential','sum'),
        totalPaymentPaid = ('payment_value','sum'),
        paymentInstallment = ('payment_installments','sum')  
)

,sequentialCount,totalPaymentPaid,paymentInstallment
order_id,,,
00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2
00018f77f2f0320c557190d7a144bdd3,1,259.83,3
000229ec398224ef6ca0657da4fc703e,1,216.87,5
00024acbcdf0a6daa1e931b038114c75,1,25.78,2
00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3
...,...,...,...
fffc94f6ce00a00581880bf54a75a037,1,343.40,1
fffcd46ef2263f404302a634eb57f7eb,1,386.53,1
fffce4705a9662cd70adb13d4a31832d,1,116.85,3


In [79]:
len(productDetails)

112650

In [80]:
# payments[payments['order_id']]

In [81]:
len(payments[payments['order_id'].isin(productDetails['order_id'])]) # product details


103056

Seller Profiling

In [82]:
sellers = dataset["sellers"]

len(sellers)

3095

In [83]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [84]:
sellers.dtypes

seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

In [85]:
sellers['seller_id'].duplicated().sum()

np.int64(0)

In [86]:
# reviews.columns
masterOrderEDA = (orderCustomerSaleEDA.merge(
                reviews[[
                    "order_id",
                    "review_score"
                ]],
                on = "order_id",
                how = "left"
))

masterOrderEDA.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,customer_state,customer_city,total_price,total_freight,total_value,item_count,purchaseMonth,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.44,-7.11,SP,sao paulo,29.99,8.72,38.71,1.00,2017-10,4.00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.78,-5.36,BA,barreiras,118.70,22.76,141.46,1.00,2018-07,4.00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.39,-17.25,GO,vianopolis,159.90,19.22,179.12,1.00,2018-08,5.00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.21,-12.98,RN,sao goncalo do amarante,45.00,27.20,72.20,1.00,2017-11,5.00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.87,-9.24,SP,santo andre,19.90,8.72,28.62,1.00,2018-02,5.00


In [87]:
payments.columns

Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='str')

In [88]:
# productDetails.columns
productDetailsEDA = (productDetails.merge(
                payments[[
                    "order_id",
                    "payment_type"
                ]],
                on = "order_id",
                how = "left"
))

In [89]:
productDetailsEDA

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,item_total,product_category_name,order_status,customer_state,customer_city,payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19,cool_stuff,delivered,RJ,campos dos goytacazes,credit_card
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83,pet_shop,delivered,SP,santa fe do sul,credit_card
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87,moveis_decoracao,delivered,MG,para de minas,credit_card
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78,perfumaria,delivered,SP,atibaia,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04,ferramentas_jardim,delivered,SP,varzea paulista,credit_card
...,...,...,...,...,...,...,...,...,...,...,...,...,...
117599,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,343.40,utilidades_domesticas,delivered,MA,sao luis,boleto
117600,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,386.53,informatica_acessorios,delivered,PR,curitiba,boleto
117601,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,116.85,esporte_lazer,delivered,SP,sao paulo,credit_card
117602,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,64.71,informatica_acessorios,delivered,SP,vinhedo,credit_card


In [90]:

productDetailsEDA.isna().sum()

order_id                 0
order_item_id            0
product_id               0
seller_id                0
shipping_limit_date      0
price                    0
freight_value            0
item_total               0
product_category_name    0
order_status             0
customer_state           0
customer_city            0
payment_type             3
dtype: int64

In [91]:
sellers.columns

Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='str')

In [92]:
masterOrderItems = (productDetailsEDA.merge(
                sellers[[
                    'seller_id',
                    'seller_city',
                    'seller_state'
                ]],
                on = 'seller_id',
                how = 'left'
))
masterOrderItems.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,item_total,product_category_name,order_status,customer_state,customer_city,payment_type,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19,cool_stuff,delivered,RJ,campos dos goytacazes,credit_card,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83,pet_shop,delivered,SP,santa fe do sul,credit_card,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87,moveis_decoracao,delivered,MG,para de minas,credit_card,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78,perfumaria,delivered,SP,atibaia,credit_card,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04,ferramentas_jardim,delivered,SP,varzea paulista,credit_card,loanda,PR


In [93]:
payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [94]:
masterOrderEDA.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 161
order_delivered_carrier_date     1793
order_delivered_customer_date    2987
order_estimated_delivery_date       0
delivery_days                    2987
delivery_delay_days              2987
customer_state                      0
customer_city                       0
total_price                       778
total_freight                     778
total_value                       778
item_count                        778
purchaseMonth                       0
review_score                      768
dtype: int64

In [101]:
# missingSummary
missingSummaryReport = pd.DataFrame(missingSummary)
missingSummaryReport

,dataset,column,missingCount,missingPercentage
0,customers,customer_id,0,0.00
1,customers,customer_unique_id,0,0.00
2,customers,customer_zip_code_prefix,0,0.00
3,customers,customer_city,0,0.00
4,customers,customer_state,0,0.00
5,geolocation,geolocation_zip_code_prefix,0,0.00
6,geolocation,geolocation_lat,0,0.00
7,geolocation,geolocation_lng,0,0.00
8,geolocation,geolocation_city,0,0.00
9,geolocation,geolocation_state,0,0.00


In [102]:
missingSummaryReport.to_csv(
    REPORTS_PATH / "missingSummaryReport.csv",
    index=False
)
print("Missing Summary report saved.")

Missing Summary report saved.


In [104]:
dataDictionaryDF.to_csv(
    REPORTS_PATH / "dataDictionary.csv",
    index = False
)
print("Data Dictionary Saved.")

Data Dictionary Saved.


In [105]:
masterOrderEDA.to_csv(
    PROCESSED_DATA_PATH / "masterOrder.csv",
    index = False
)
print("Master Order Data Saved.")

Master Order Data Saved.


In [106]:
masterOrderItems.to_csv(
    PROCESSED_DATA_PATH / "masterOrderItems.csv",
    index = False
)
print("Master Order Items Data Saved.")

Master Order Items Data Saved.


np.int64(0)